In [2]:
!sudo apt-get update && sudo apt-get install -y espeak-ng
!pip install coqui-tts datasets soundfile

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:6 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,452 kB]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:11 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,794 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:13 https://ppa.launchpadcontent.net/deadsnakes

In [4]:
import os
import pandas as pd
import soundfile as sf
import librosa
import torch
from huggingface_hub import snapshot_download
from google.colab import userdata


from TTS.tts.configs.shared_configs import BaseAudioConfig, BaseDatasetConfig
from TTS.tts.configs.vits_config import VitsConfig
from TTS.utils.audio import AudioProcessor
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.vits import Vits
from TTS.tts.utils.text.tokenizer import TTSTokenizer
from trainer import Trainer, TrainerArgs
from TTS.api import TTS



# ==========================================
# 0. CONFIG
# ==========================================
HF_TOKEN  = userdata.get("HF_TOKEN")

REPO_ID        = "ankitdhiman/haryanvi-tts"
TARGET_SR      = 22050
DATA_DIR       = "haryanvi_vits_data"
WAV_DIR        = os.path.join(DATA_DIR, "wavs")
LOCAL_REPO     = "hf_dataset_cache"
PHONEME_CACHE  = os.path.join(DATA_DIR, "phoneme_cache")
TEXT_MIN_WORDS = 3
TEXT_MAX_WORDS = 25

os.makedirs(WAV_DIR,       exist_ok=True)
os.makedirs(PHONEME_CACHE, exist_ok=True)



# ==========================================
# 1. DOWNLOAD (skips if already present)
# ==========================================
_marker = os.path.join(LOCAL_REPO, "metadata.csv")

if os.path.exists(_marker):
    print(f"✅ Dataset already downloaded at '{LOCAL_REPO}' — skipping download.")
    local_repo = LOCAL_REPO
else:
    print("--- Downloading full dataset from HuggingFace ---")
    local_repo = snapshot_download(
        repo_id=REPO_ID,
        repo_type="dataset",
        token=HF_TOKEN,
        local_dir=LOCAL_REPO,
    )
    print(f"  Dataset downloaded to: {local_repo}")



# ==========================================
# 2. READ & PREPROCESS METADATA
# ==========================================
print("--- Reading and preprocessing metadata ---")

df_meta = pd.read_csv(os.path.join(local_repo, "metadata.csv"))
print(f"  Raw shape      : {df_meta.shape}")

# Drop rows with missing text
df_meta = df_meta.dropna(subset=["text"])
df_meta = df_meta[df_meta["text"].str.strip().str.lower() != "nan"]

# Deduplicate on text
df_meta = df_meta.drop_duplicates(subset="text")
print(f"  After dedup    : {df_meta.shape}")

# Word-count filter: keep only 3–25 word sentences
df_meta = df_meta[
    df_meta["text"].apply(lambda x: TEXT_MIN_WORDS <= len(str(x).split()) <= TEXT_MAX_WORDS)
].reset_index(drop=True)
print(f"  After filtering: {df_meta.shape}")

print(f"  Columns        : {df_meta.columns.tolist()}")



# ==========================================
# 3. PROCESS AUDIO FILES
# ==========================================
_out_meta = os.path.join(DATA_DIR, "metadata.csv")

if os.path.exists(_out_meta):
    print(f"✅ Processed wavs already exist at '{DATA_DIR}' — skipping processing.")
else:
    print("--- Processing audio files ---")
    metadata_ljspeech = []
    skipped = 0

    for idx, row in df_meta.iterrows():
        rel_path = str(row["file_name"]).strip()
        text_val = str(row["text"]).strip()

        audio_abs_path = os.path.join(local_repo, rel_path)

        if not os.path.exists(audio_abs_path):
            print(f"  [{idx}] SKIP — file not found: {audio_abs_path}")
            skipped += 1
            continue

        try:
            audio_array, _ = librosa.load(audio_abs_path, sr=TARGET_SR, mono=True)
        except Exception as e:
            print(f"  [{idx}] SKIP — decode error: {e}")
            skipped += 1
            continue

        file_name = f"audio_{idx}"
        sf.write(os.path.join(WAV_DIR, f"{file_name}.wav"), audio_array, TARGET_SR)
        metadata_ljspeech.append(f"{file_name}|{text_val}|{text_val}")

    assert len(metadata_ljspeech) > 0, (
        f"❌ No samples exported! {skipped} skipped."
    )

    with open(_out_meta, "w", encoding="utf-8") as f:
        f.write("\n".join(metadata_ljspeech))

    print(f"✅ Exported {len(metadata_ljspeech)} samples | Skipped: {skipped}")



# ==========================================
# 4. VITS CONFIGURATION
# ==========================================
dataset_config = BaseDatasetConfig(
    formatter="ljspeech",
    meta_file_train="metadata.csv",
    path=DATA_DIR
)

audio_config = BaseAudioConfig(
    sample_rate=TARGET_SR,
    win_length=1024,
    hop_length=256,
    fft_size=1024,
    num_mels=80,
    mel_fmin=0.0,
    mel_fmax=None,
    preemphasis=0.0,
    ref_level_db=20,
    min_level_db=-100,
    do_trim_silence=True,
    trim_db=45,
)

config = VitsConfig(
    audio=audio_config,
    run_name="vits_haryanvi",
    batch_size=8,
    eval_batch_size=4,
    num_loader_workers=2,
    num_eval_loader_workers=2,
    run_eval=True,
    test_delay_epochs=-1,
    epochs=1,
    text_cleaner="basic_cleaners",
    use_phonemes=False,
    compute_input_seq_cache=True,
    phoneme_cache_path=PHONEME_CACHE,
    print_step=25,
    datasets=[dataset_config],
)



# ==========================================
# 5. INITIALIZE MODEL & TRAINER
# ==========================================
print("\n--- Initializing VITS ---")

ap = AudioProcessor.init_from_config(config)

train_samples, eval_samples = load_tts_samples(
    dataset_config,
    eval_split=True,
    eval_split_max_size=50,
    eval_split_size=0.1
)

print(f"  Train samples : {len(train_samples)}")
print(f"  Eval  samples : {len(eval_samples)}")

tokenizer, config = TTSTokenizer.init_from_config(config)
model   = Vits(config, ap, tokenizer, speaker_manager=None)
trainer = Trainer(
    TrainerArgs(),
    config,
    output_path="tts_output",
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples
)

print("\n--- Starting Training ---")
trainer.fit()





# ==========================================
# 6. INFERENCE
# ==========================================
print("\n--- Running Inference ---")

import glob

output_dirs = sorted(
    [os.path.join("tts_output", d) for d in os.listdir("tts_output") if config.run_name in d],
    key=os.path.getmtime
)
latest_run_dir = output_dirs[-1]
config_path    = os.path.join(latest_run_dir, "config.json")

# Prefer best_model.pth, fall back to latest checkpoint by modified time
pth_files  = glob.glob(os.path.join(latest_run_dir, "*.pth"))
assert len(pth_files) > 0, f"❌ No .pth checkpoint found in: {latest_run_dir}"

best = [f for f in pth_files if "best_model" in os.path.basename(f)]
checkpoint_path = best[0] if best else sorted(pth_files, key=os.path.getmtime)[-1]

print(f"  Using checkpoint: {os.path.basename(checkpoint_path)}")

tts = TTS(
    model_path=checkpoint_path,
    config_path=config_path,
    progress_bar=False,
    gpu=torch.cuda.is_available()
)

test_sentence = "मैं आज बाजार जा रया सै।"
output_file   = "test_haryanvi_output.wav"
tts.tts_to_file(text=test_sentence, file_path=output_file)

print(f"✅ Done! Audio saved to: {output_file}")



✅ Dataset already downloaded at 'hf_dataset_cache' — skipping download.
--- Reading and preprocessing metadata ---
  Raw shape      : (2768, 2)
  After dedup    : (2767, 2)
  After filtering: (2764, 2)
  Columns        : ['file_name', 'text']
✅ Processed wavs already exist at 'haryanvi_vits_data' — skipping processing.

--- Initializing VITS ---
  Train samples : 2714
  Eval  samples : 50


 > Training Environment:
 | > Backend: Torch
 | > Mixed precision: False
 | > Precision: float32
 | > Current device: 0
 | > Num. of GPUs: 1
 | > Num. of CPUs: 2
 | > Num. of Torch Threads: 1
 | > Torch seed: 54321
 | > Torch CUDNN: True
 | > Torch CUDNN deterministic: False
 | > Torch CUDNN benchmark: False
 | > Torch TF32 MatMul: False
 > Start Tensorboard: tensorboard --logdir=tts_output/vits_haryanvi-March-19-2026_07+52PM-0000000

 > Model has 83046892 parameters

 > EPOCH: 0/0
 --> tts_output/vits_haryanvi-March-19-2026_07+52PM-0000000



--- Starting Training ---



 > TRAINING (2026-03-19 19:52:40) 

   --> TIME: 2026-03-19 19:52:42 -- STEP: 0/339 -- GLOBAL_STEP: 0
     | > current_lr_0: 0.0002  (0.0002)
     | > current_lr_1: 0.0002  (0.0002)
     | > loss_disc: 5.992541313171387  (5.992541313171387)
     | > loss_disc_real_0: 1.0660340785980225  (1.0660340785980225)
     | > loss_disc_real_1: 0.9701231122016907  (0.9701231122016907)
     | > loss_disc_real_2: 0.9970991611480713  (0.9970991611480713)
     | > loss_disc_real_3: 0.9724589586257935  (0.9724589586257935)
     | > loss_disc_real_4: 0.9755966067314148  (0.9755966067314148)
     | > loss_disc_real_5: 1.0095603466033936  (1.0095603466033936)
     | > loss_0: 5.992541313171387  (5.992541313171387)
     | > grad_norm_0: tensor(6.6705, device='cuda:0')  (tensor(6.6705, device='cuda:0'))
     | > loss_gen: 4.55568790435791  (4.55568790435791)
     | > loss_kl: 217.56993103027344  (217.56993103027344)
     | > loss_feat: 0.6320661306381226  (0.6320661306381226)
     | > loss_mel: 121.416297


--- Running Inference ---
  Using checkpoint: best_model_340.pth


/usr/local/lib/python3.12/dist-packages/TTS/api.py:93: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")


✅ Done! Audio saved to: test_haryanvi_output.wav
